In [4]:
# LECTURA DEL EXCEL train_test_errors-summary.xls
import pandas as pd
import numpy as np

# Install openpyxl if you haven't already
!pip install openpyxl

try:
  df = pd.read_excel('train_test_errors-summary.xlsx')
  print(df)
except FileNotFoundError:
  print("Error: 'train_test_errors-summary.xls' not found. Please upload the file.")
except Exception as e:
  print(f"An error occurred: {e}")

       Model     Error Dataset
0        PLS  0.612400   Train
1        PLS -0.130964   Train
2        PLS -0.158878   Train
3        PLS -0.143950   Train
4        PLS -0.111857   Train
..       ...       ...     ...
483  PLS-VIP -0.373613    Test
484  PLS-VIP -0.597752    Test
485  PLS-VIP  0.369349    Test
486  PLS-VIP -0.050659    Test
487  PLS-VIP -0.285236    Test

[488 rows x 3 columns]


In [3]:
  # Filter for Dataset == 'Test'
  df_test = df[df['Dataset'] == 'Test']

  # Drop the 'Dataset' column
  df_test = df_test.drop(columns=['Dataset'])
  df_test['Absolute Error'] = df_test['Error'].abs()

  print(df_test)



       Model     Error  Absolute Error
92       PLS -0.690589        0.690589
93       PLS -0.414330        0.414330
94       PLS  0.334322        0.334322
95       PLS  0.785734        0.785734
96       PLS  0.325836        0.325836
..       ...       ...             ...
483  PLS-VIP -0.373613        0.373613
484  PLS-VIP -0.597752        0.597752
485  PLS-VIP  0.369349        0.369349
486  PLS-VIP -0.050659        0.050659
487  PLS-VIP -0.285236        0.285236

[120 rows x 3 columns]


In [11]:
import pandas as pd
import scipy.stats as stats
from itertools import combinations


# Obtener los grupos únicos
grupos = df_test['Model'].unique()

# Generar todas las combinaciones de pares de grupos
pares_grupos = list(combinations(grupos, 2))

# Lista para almacenar los resultados
resultados = []

# Realizar las pruebas pareadas para cada par de grupos
for g1, g2 in pares_grupos:
    # Filtrar los valores de error para los dos grupos
    errores_g1 = df_test[df_test['Model'] == g1]['Absolute Error']
    errores_g2 = df_test[df_test['Model'] == g2]['Absolute Error']


    # Verificar que ambos grupos tengan el mismo tamaño
    if len(errores_g1) == len(errores_g2):
        # Prueba t pareada
        #t_stat, p_value = stats.ttest_rel(errores_g1, errores_g2)
        #t_stat, p_value = stats.wilcoxon(errores_g1, errores_g2)
        t_stat, p_value_t=stats.ttest_rel(errores_g1, errores_g2)
        w_stat, p_value_w=stats.wilcoxon(errores_g1, errores_g2)
        resultados.append({
            'Group 1': g1,
            'Group 2': g2,
            'Paired sample t-test': p_value_t,
            'Wilcoxon test': p_value_w
        })
    else:
        # Si los tamaños no coinciden, registrar un mensaje
        resultados.append({
            'Group 1': g1,
            'Group 2': g2,
            'Paired sample t-test': None,
            'Wilcoxon test': None,
            'Comentario': 'Tamaños de grupo desiguales'
        })

# Convertir los resultados a un DataFrame
resultados_df = pd.DataFrame(resultados)

# Mostrar los resultados
print(resultados_df)


    Group 1   Group 2  Paired sample t-test  Wilcoxon test
0       PLS   PLS-BVE              0.700934       0.289366
1       PLS  PLS-GSMW              0.296081       0.212933
2       PLS   PLS-VIP              0.423427       0.318396
3   PLS-BVE  PLS-GSMW              0.618577       0.289366
4   PLS-BVE   PLS-VIP              0.556346       0.502761
5  PLS-GSMW   PLS-VIP              0.796435       0.427955
